# NotebookLM podcast from links (Gemini Notebook API)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/useapi/notebooklm-api/blob/main/notebooks/notebooklm_podcast.ipynb)

Turn web pages and YouTube videos into a NotebookLM **Audio Overview** (the two-host podcast), then play it right here. It uses the [Gemini Notebook API](https://useapi.net/docs/api-gemini-notebook-v1?utm_source=github&utm_medium=colab&utm_campaign=notebooklm-colab) by useapi.net, a third-party REST API that runs on your own Google NotebookLM account (NotebookLM is now called Gemini Notebook).

You need:
1. A useapi.net [API token](https://useapi.net/docs/start-here/setup-useapi?utm_source=github&utm_medium=colab&utm_campaign=notebooklm-colab). One [$15/month subscription](https://useapi.net/docs/subscription?utm_source=github&utm_medium=colab&utm_campaign=notebooklm-colab) covers every useapi.net API.
2. A Google account connected with the [Gemini Notebook setup](https://useapi.net/docs/start-here/setup-gemini-notebook?utm_source=github&utm_medium=colab&utm_campaign=notebooklm-colab). Generation runs on that account's own NotebookLM plan, the free one included.

Run the cells from top to bottom (**Runtime → Run all**). A short brief takes about 3 to 10 minutes.

## 1. Your token and Google account

In [ ]:
from getpass import getpass

# Paste your useapi.net API token when asked (it is not saved in the notebook)
USEAPI_TOKEN = getpass('useapi.net API token: ')
# Optional: the connected Google account to use. Leave empty and the API uses the least busy healthy one.
EMAIL = ''  # e.g. 'you@gmail.com'

## 2. Sources and podcast options

In [ ]:
TITLE = 'Apollo 11: the landing that nearly aborted'
URLS = [
    'https://en.wikipedia.org/wiki/Apollo_11',
    'https://www.youtube.com/watch?v=xUcYQ7slmRw',  # YouTube links are read from their captions
]
FORMAT = 'brief'     # deep_dive, brief, critique, debate
LENGTH = 'short'     # short, default, long
LANGUAGE = 'en'      # en, es, pt_BR, zh_Hans, ...
INSTRUCTIONS = ''    # optional, e.g. 'Focus on the computer alarms'

## 3. Create a notebook and add the sources

In [ ]:
import time, requests

API = 'https://api.useapi.net/v1/gemini-notebook'
session = requests.Session()
session.headers['Authorization'] = f'Bearer {USEAPI_TOKEN}'


def call(method, path, **kwargs):
    r = session.request(method, f'{API}/{path}', timeout=320, **kwargs)
    if not r.ok:
        raise RuntimeError(f'{method} /{path} -> HTTP {r.status_code}: {r.text}')
    return r.json()


body = {'title': TITLE}
if EMAIL:
    body['email'] = EMAIL
notebook = call('POST', 'notebooks', json=body)['notebook']
print('notebook', notebook)

for s in call('POST', 'sources', json={'notebook': notebook, 'urls': URLS})['sources']:
    print(f"  {s['kind']:8} {s['status']:10} {s['title']}")

# Google fetches and indexes every source before it can be used (give up after 10 minutes)
for _ in range(60):
    sources = call('GET', f'notebooks/{requests.utils.quote(notebook, safe="")}')['sources']
    statuses = [s['status'] for s in sources]
    if all(st in ('ready', 'error') for st in statuses):
        break
    print('  sources:', ', '.join(statuses))
    time.sleep(10)
else:
    raise RuntimeError(f"Sources still processing after 10 minutes: {', '.join(statuses)}. Remove the stuck link from URLS and run this cell again.")
print('sources ready:', sum(st == 'ready' for st in statuses), 'of', len(statuses))

## 4. Generate the Audio Overview and wait for it

In [ ]:
audio = {'notebook': notebook, 'type': 'audio', 'mode': 'async',
         'format': FORMAT, 'length': LENGTH, 'language': LANGUAGE}
if INSTRUCTIONS:
    audio['instructions'] = INSTRUCTIONS
jobid = call('POST', 'artifacts', json=audio)['jobid']
print('audio job', jobid)

start = time.time()
while True:
    job = call('GET', f'jobs/{requests.utils.quote(jobid, safe="")}')
    if job['status'] in ('completed', 'failed'):
        break
    print(f"  {job['status']} ({time.time() - start:.0f} s)")
    time.sleep(15)
if job['status'] == 'failed':
    raise RuntimeError(f"Audio Overview failed: {job.get('error')}")
result = job['result']
print(f"done in {time.time() - start:.0f} s: \"{result['title']}\", {result.get('duration', '?')} s of audio")

## 5. Download, play and save it

The download link needs your token, so the file is fetched here and played from memory.

In [ ]:
import re

file = next(f for f in result['files'] if f['format'] == 'm4a')
r = session.get(file['url'], timeout=300)
r.raise_for_status()
audio_bytes = r.content
filename = re.sub(r'[^\w-]+', '_', result['title'])[:60] + '.m4a'
with open(filename, 'wb') as out:
    out.write(audio_bytes)
print(f'saved {filename} ({len(audio_bytes) / 1e6:.1f} MB)')

try:
    from IPython.display import Audio, display
    player = Audio(filename=filename)
    player.mimetype = 'audio/mp4'  # an .m4a file
    display(player)
except ImportError:
    pass  # not in a notebook

In [ ]:
# Save the file to your computer (Colab only)
try:
    from google.colab import files
    files.download(filename)
except ImportError:
    print(f'Not on Colab: the file is at {filename}')

## Next steps

- The notebook stays in your NotebookLM account. Generate a Video Overview, slides or a report from it with [POST /artifacts](https://useapi.net/docs/api-gemini-notebook-v1/post-gemini-notebook-artifacts?utm_source=github&utm_medium=colab&utm_campaign=notebooklm-colab) (`type: "video"`, `"slides"`, `"report"`...).
- Making many podcasts? Connect several Google accounts with [POST /accounts](https://useapi.net/docs/api-gemini-notebook-v1/post-gemini-notebook-accounts?utm_source=github&utm_medium=colab&utm_campaign=notebooklm-colab) and leave `EMAIL` empty: new jobs are spread across them automatically.
- More examples (Node.js, Python, bash, an agent skill and n8n templates): [github.com/useapi/notebooklm-api](https://github.com/useapi/notebooklm-api).